# FinRAG — Qwen Server on Colab GPU

This notebook loads the fine-tuned Qwen 2.5 7B model and serves it over a public tunnel so your local Streamlit app can call it.
Use a **T4 GPU** runtime: Runtime → Change runtime type → T4 GPU.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > GPU before running.'
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Clone the project from GitHub. Drive is still mounted above because the LoRA adapter is saved there.

In [ ]:
import subprocess, os

REPO_URL    = 'https://github.com/pendyal1/Financial_QA_ChatBot.git'
BRANCH      = 'integration/finrag-final'
PROJECT_DIR = '/content/Financial_QA_ChatBot'

if not os.path.exists(PROJECT_DIR):
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, PROJECT_DIR], check=True)
    print('Cloned.')
else:
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull'], check=True)
    print('Pulled latest.')

%cd $PROJECT_DIR

In [ ]:
!bash scripts/colab_setup.sh
# If this reports mixed CUDA versions: Runtime > Disconnect and delete runtime, then reconnect.

---
## Fine-Tuning (skip if adapter already trained)

Run these three cells only if you need to (re)train the LoRA adapter. Otherwise jump straight to the **Serve** section below.

In [ ]:
!PYTHONPATH=src python -m finrag.fine_tuning --limit 2000

In [ ]:
!PYTHONPATH=src python -m finrag.train_qlora   --model-name Qwen/Qwen2.5-7B-Instruct   --train-file data/fine_tuning/finqa_train.jsonl   --output-dir "/content/drive/MyDrive/Generative AI Project FinRAG/finrag_lora_adapter"   --epochs 1   --max-length 1536   --batch-size 1   --gradient-accumulation-steps 16   --learning-rate 2e-4   --lora-r 16   --lora-alpha 32

In [ ]:
!PYTHONPATH=src python -m finrag.hf_adapter_answer   --adapter-path "/content/drive/MyDrive/Generative AI Project FinRAG/finrag_lora_adapter"   --model-name Qwen/Qwen2.5-7B-Instruct   "What risks did Apple report related to supply chains?"

---
## Serve Qwen from the Colab GPU

Run these cells to start the server and expose it publicly. Then paste the tunnel URL into your local Streamlit sidebar.

In [ ]:
import subprocess, os, time

ADAPTER = "/content/drive/MyDrive/Generative AI Project FinRAG/finrag_lora_adapter"

# Kill any existing server instance
subprocess.run(["pkill", "-f", "finrag.qwen_server"], capture_output=True)

cmd = [
    "python", "-m", "finrag.qwen_server",
    "--model-name", "Qwen/Qwen2.5-7B-Instruct",
    "--port", "8000",
]
if os.path.isdir(ADAPTER):
    cmd += ["--adapter-path", ADAPTER]
    print("Adapter found — loading fine-tuned model.")
else:
    print(f"Adapter not found at:
  {ADAPTER}
Loading base model only.")

env = {**os.environ, "PYTHONPATH": "src"}
log = open("qwen_server.log", "w")
subprocess.Popen(cmd, env=env, stdout=log, stderr=log)

print("Server starting... waiting 10s before checking log.")
time.sleep(10)
with open("qwen_server.log") as f:
    print(f.read())

In [ ]:
%%bash
# Poll until the server responds on /health (model loading takes 1-2 min).
echo "Waiting for server to be ready..."
for i in $(seq 1 30); do
    if curl -s http://127.0.0.1:8000/health | grep -q '"ok"'; then
        echo "Server is up!"
        curl -s http://127.0.0.1:8000/health
        exit 0
    fi
    echo "  attempt $i/30 — sleeping 10s..."
    sleep 10
done
echo "Server did not start in time. Check qwen_server.log:"
tail -60 qwen_server.log

In [ ]:
# Optional: smoke-test the /hallucinate endpoint.
# The NLI model (~450 MB) downloads on first call — takes ~30s.
import requests

payload = {
    "answer": "Apple reported revenue of $391 billion in fiscal year 2024.",
    "passages": [{
        "chunk_id": "test-1",
        "score": 0.9,
        "ticker": "AAPL",
        "company": "Apple Inc.",
        "source": "10-K 2024",
        "source_url": "https://www.sec.gov/",
        "text": (
            "Net sales for fiscal 2024 totaled $391.0 billion, compared to "
            "$383.3 billion in fiscal 2023, an increase of 2 percent year over year."
        ),
    }],
}

resp = requests.post("http://127.0.0.1:8000/hallucinate", json=payload, timeout=180)
resp.raise_for_status()
result = resp.json()
print("overall_risk   :", result["overall_risk"])
print("confidence     :", result["confidence_score"])
for c in result["claims"]:
    print(f"  [{c['label']}] {c['claim_text']}")

### Expose the server publicly

Run the Cloudflare cell below to get a public URL. Copy it and paste into the **Colab Qwen endpoint URL** field in the Streamlit sidebar.

In [ ]:
%%bash
pkill -f cloudflared || true
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
chmod +x cloudflared
nohup ./cloudflared tunnel --url http://127.0.0.1:8000 > cloudflared.log 2>&1 &
sleep 8
grep -o 'https://[-a-zA-Z0-9.]*trycloudflare.com' cloudflared.log | tail -1

### Fallback: ngrok tunnel

Use this if Cloudflare returns a 500 error. Get a free auth token from https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
from pyngrok import ngrok

NGROK_AUTHTOKEN = ''  # paste your ngrok token here
if not NGROK_AUTHTOKEN:
    raise ValueError('Paste your ngrok auth token into NGROK_AUTHTOKEN')

ngrok.set_auth_token(NGROK_AUTHTOKEN)
public_url = ngrok.connect(8000, 'http').public_url
print(public_url)